# 05 - Managed vs External Tables

This notebook explores the lifecycle and data protection differences between Managed and External tables in Unity Catalog.

### Key Concepts Covered:
* Creating a Managed Table in `aura_gold` schema
* Checking its physical location via `DESCRIBE DETAIL`
* Dropping the managed table and observing the complete deletion of both metadata and physical files in storage.


## Step 1: Switch Context to aura_gold Schema


In [0]:
%sql
USE CATALOG demoworkspace2;
USE SCHEMA aura_gold;


## Step 2: Create a Managed Table


In [0]:
%sql
DROP TABLE IF EXISTS sales_summary;

CREATE TABLE sales_summary (
    store_id STRING,
    total_quantity BIGINT,
    total_amount DOUBLE
);


## Step 3: Populate Managed Table


In [0]:
%sql
INSERT INTO sales_summary VALUES
('STR-101', 3, 1349.98),
('STR-102', 17, 107.95);


## Step 4: Describe Table Details to Find Cloud Storage Path


In [0]:
%sql
DESCRIBE DETAIL sales_summary;


## Step 5: Check Files in Storage (Python Cell)


In [0]:
table_info = spark.sql("DESCRIBE DETAIL sales_summary").collect()[0]
table_path = table_info.location
print(f"Checking storage path: {table_path}")
display(dbutils.fs.ls(table_path))


## Step 6: Drop the Managed Table


In [0]:
%sql
DROP TABLE sales_summary;


## Step 7: Verify Table Metadata is Gone


In [0]:
%sql
SHOW TABLES;


## Step 8: Verify Physical Files are Deleted (Python Cell)


In [0]:
try:
    dbutils.fs.ls(table_path)
    print("WARNING: Files still exist in storage!")
except Exception as e:
    print("SUCCESS: The storage location no longer exists! Unity Catalog successfully deleted all data files.")
